<a href="https://colab.research.google.com/github/andygaosure/Small-Group-Regression-Competition/blob/main/Copy_of_kaggle_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [ ]:
#download train.csv
house = pd.read_csv("/content/sample_data/train.csv")
house_test = pd.read_csv("/content/sample_data/test.csv")
house.head()
house_test.head()
print(house.info())
house.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1200 non-null   int64  
 1   MSSubClass     1200 non-null   int64  
 2   MSZoning       1200 non-null   object 
 3   LotFrontage    990 non-null    float64
 4   LotArea        1200 non-null   int64  
 5   Street         1200 non-null   object 
 6   Alley          75 non-null     object 
 7   LotShape       1200 non-null   object 
 8   LandContour    1200 non-null   object 
 9   Utilities      1200 non-null   object 
 10  LotConfig      1200 non-null   object 
 11  LandSlope      1200 non-null   object 
 12  Neighborhood   1200 non-null   object 
 13  Condition1     1200 non-null   object 
 14  Condition2     1200 non-null   object 
 15  BldgType       1200 non-null   object 
 16  HouseStyle     1200 non-null   object 
 17  OverallQual    1200 non-null   int64  
 18  OverallC

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Filtering out data that has more than 100 N/As. We believe that columns with more than this cannot be reliably replaced with estimated/averaged values.

In [ ]:
x = len(house.columns)-1
house_new = house
while x >= 0:
  if house[house.columns[x]].isna().sum() > 100:
    print(house.columns[x])
    house_new = house_new.drop(house.columns[x], axis=1)
  x-=1
house_new.shape

MiscFeature
Fence
PoolQC
FireplaceQu
MasVnrType
Alley
LotFrontage


(1200, 74)

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

This was testing out a simple regression model. Not important to final submission

In [ ]:
X = house_new[["LotArea"]]
Y = house_new["SalePrice"]
linregsimple = LinearRegression()
linregsimple.fit(X,Y)
print(linregsimple.intercept_)
print(linregsimple.coef_)
print(root_mean_squared_error(Y,linregsimple.predict(X)))

159905.66203706482
[2.03694741]
78099.43204969873


This was testing how to count the number of unique values within a column for the while loop. There probably was an easier way to go about this

In [ ]:
#np.unique(house_new[63]).tolist()
len(np.unique(house[house.columns[63]].value_counts()).tolist())

4

We looked for columns with more than 2 unique values (2 means it can be OneHotEncoded) and less than 10 (10 or more implies that the data is quantitative). It's not perfect but it gives us an idea of what the ordinal columns are. Out of these, we picked ones that we believed were important

In [ ]:
oink = len(house_new.columns) - 1
while oink > 0:
  if len(np.unique(house[house.columns[oink-1]].value_counts()).tolist()) > 2 and len(np.unique(house[house.columns[oink-1]].value_counts()).tolist()) < 10:
    print(house_new.columns[oink])
  oink = oink-1

house_ordinal = house_new[['SaleType', 'GarageQual', 'GarageCars', 'Fireplaces', 'OverallCond', 'OverallQual']]

SaleType
YrSold
MoSold
ScreenPorch
3SsnPorch
EnclosedPorch
WoodDeckSF
PavedDrive
GarageQual
GarageArea
GarageCars
GarageFinish
GarageType
Fireplaces
Functional
TotRmsAbvGrd
KitchenQual
KitchenAbvGr
BedroomAbvGr
FullBath
BsmtHalfBath
GrLivArea
2ndFlrSF
1stFlrSF
CentralAir
HeatingQC
Heating
TotalBsmtSF
BsmtUnfSF
BsmtFinSF2
BsmtFinType2
BsmtFinSF1
BsmtFinType1
BsmtExposure
BsmtCond
BsmtQual
Foundation
MasVnrArea
Exterior2nd
RoofStyle
YearBuilt
OverallCond
OverallQual
HouseStyle
Condition2
Condition1
LandSlope
LotConfig
LotArea


In [ ]:
house_new['Condition2'].sort_values().unique()
house_new['OverallQual'].sort_values().unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=house_ordinal)

We realized that some of the columns we picked were actual rankings, not just columns that needed to be ordinally ranked. The two columns that were truly ordinal are ranked here (though this and the next code cell are only for testing)

In [ ]:
#ordinal columns

import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

missing = ['SaleType', 'GarageQual']
for x in missing:
    house_ordinal[x] = house_ordinal[x].fillna('0')

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

house_ordinal1 = house_ordinal.copy()
house_ordinal1[house_ordinal.columns] = oe.fit_transform(house_ordinal)
house_ordinal.info(), house_ordinal1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   SaleType     1200 non-null   object
 1   GarageQual   1200 non-null   object
 2   GarageCars   1200 non-null   int64 
 3   Fireplaces   1200 non-null   int64 
 4   OverallCond  1200 non-null   int64 
 5   OverallQual  1200 non-null   int64 
dtypes: int64(4), object(2)
memory usage: 56.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   SaleType     1200 non-null   float64
 1   GarageQual   1200 non-null   float64
 2   GarageCars   1200 non-null   float64
 3   Fireplaces   1200 non-null   float64
 4   OverallCond  1200 non-null   float64
 5   OverallQual  1200 non-null   float64
dtypes: float64(6)
memory usage: 56.4 KB


/tmp/ipykernel_7124/3386303975.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  house_ordinal[x] = house_ordinal[x].fillna('0')


(None, None)

Selecting quantitative data (we used basically all of them)

In [ ]:
# quantitative data
house_q = house.select_dtypes(include=['int64','float'])
house_q.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 38 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1200 non-null   int64  
 1   MSSubClass     1200 non-null   int64  
 2   LotFrontage    990 non-null    float64
 3   LotArea        1200 non-null   int64  
 4   OverallQual    1200 non-null   int64  
 5   OverallCond    1200 non-null   int64  
 6   YearBuilt      1200 non-null   int64  
 7   YearRemodAdd   1200 non-null   int64  
 8   MasVnrArea     1194 non-null   float64
 9   BsmtFinSF1     1200 non-null   int64  
 10  BsmtFinSF2     1200 non-null   int64  
 11  BsmtUnfSF      1200 non-null   int64  
 12  TotalBsmtSF    1200 non-null   int64  
 13  1stFlrSF       1200 non-null   int64  
 14  2ndFlrSF       1200 non-null   int64  
 15  LowQualFinSF   1200 non-null   int64  
 16  GrLivArea      1200 non-null   int64  
 17  BsmtFullBath   1200 non-null   int64  
 18  BsmtHalf

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge

#defining ordinal, not ordinal (but ranked), and quantitative columns
house_ordinal = ['SaleType', 'GarageQual']
house_ordinal_jk = ['GarageCars', 'Fireplaces', 'OverallCond', 'OverallQual']
values = [z for z in house_q.columns if z not in ['SalePrice', 'Id'] + house_ordinal_jk]

#building pipeline to impute N/As and standardize the quantitative columns
nums = Pipeline([('imputer', SimpleImputer(strategy='mean')),('scaler', StandardScaler())])

#pipeimpute is for the ordinal columns
pipeimpute = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', fill_value='most_frequent')),('encoder', OrdinalEncoder())])

#making a column transformer by combining the two pipelines. the four columns of house_ordinal_jk don't need to be imputed or manipulated
transform = make_column_transformer((pipeimpute, house_ordinal),(nums, values),('passthrough', house_ordinal_jk))

#ridge regression actually reduces RMSE by a little bit compared to linear regression, so we used that
pipe = Pipeline([('transformer', transform),('Ridge', Ridge())])
X_train = house[values + house_ordinal + house_ordinal_jk]
y_train = house['SalePrice']

#fitting data
pipe.fit(X_train, y_train)
print(root_mean_squared_error(y_train, pipe.predict(X_train)))
pipe

#creating csv file for submission
preds = pipe.predict(house_test[values + house_ordinal + house_ordinal_jk])
pdf = pd.DataFrame(preds, columns = ['SalePrice'])
#we suspect that the id's not needing to be subtracted by 1201 may be a source of error. predicted values may not line up properly
pdf['Id'] = house_test['Id']-1201
pdf = pdf[['Id', 'SalePrice']]
pdf.to_csv('preds2.csv', index=False)

31642.507789382413
